# Himawari-9 over Anak Krakatau - 2. HSD to map-projected frames

Benny Istanto - https://benny.istan.to

---

HSD is raw instrument geometry: counts on Himawari-9's fixed scanning grid,
one file per band per segment. Getting from there to a frame means
calibrating to radiance and brightness temperature, building an RGB recipe
from several bands, and reprojecting onto a stable map grid so features stay
put between frames instead of sliding around.

[Satpy](https://satpy.readthedocs.io) does all three. This notebook wires it
up and renders the products listed in `config.yml`.

**The composites**

| Composite | Channels | Day | Night | What it shows |
|---|---|---|---|---|
| `true_color` | B01-B04, sharpened to 0.5 km | yes | no | what the eye would see - the brown plume against ocean |
| `ash` | B11, B13, B14, B15 | yes | yes | volcanic ash, distinct from meteorological cloud |
| `dust` | same four | yes | yes | same channels, stretch tuned for lofted dust and SO2 |
| `night_microphysics` | B07, B13, B15 | yes | yes | cloud phase and plume top |
| `volcanic_emissions` | B09, B10, B11, B13 | yes | yes | SO2 RGB - added by this project, see below |

The IR products are the ones that matter for an eruption timeline: they work
straight through the night, so a three-day animation has no day/night seam.

**One composite satpy does not ship.** Satpy defines the CIRA SO2 RGB for ABI
and FCI but not for AHI. `config/satpy/` adds it, transcribed to the matching
AHI channels, together with its enhancement - the stretch is not optional, as
without it the composite falls through to a default linear scaling and looks
nothing like the reference product. `use_local_config()` puts that directory
on satpy's search path, where it is *merged* with the built-in AHI set rather
than replacing it.

## 1. Configuration

In [ ]:
import sys, warnings
from pathlib import Path

# The package lives in src/ and is not installed, so put it on the path.
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

from himawari import project

cfg = project.load(ROOT / "config.yml")
cfg.make_dirs()
print(cfg.summary())

from himawari import render as R

R.use_local_config()          # registers the SO2 composite + its enhancement

# Which bands each composite needs is a static table in `catalog.py`, because
# the download stage has to know it without importing satpy. Static tables
# drift, and a wrong one fails in a thoroughly unhelpful way: satpy logs
# "Required file type 'hsd_bNN' not found", returns an empty Scene, and the
# error you actually see is "No dataset areas available" from the cropper.
#
# So check the table against satpy's own composite definitions up front. This
# resolves each composite's dependency tree properly - through nested
# compositors, and through prerequisites stated by wavelength rather than by
# name, which is how true_color picks up B02 and B04 via hybrid_green.
drift = R.verify_requirements(cfg.composites)
if drift:
    for name, (needs, table) in drift.items():
        print(f"MISMATCH {name}: satpy needs {needs}, table says {table}")
    raise SystemExit("catalog.COMPOSITE_REQUIREMENTS is out of date - fix it before rendering")
print("band requirements verified against satpy")
print()

for p in cfg.products:
    print(p)

## 2. Coastlines, drawn once

The output grid is plate carree, so a (lon, lat) vertex maps to a pixel with
one linear scale per axis. That makes it worth pre-rendering the coastlines
and graticule into a transparent PNG at exactly the grid's pixel dimensions,
once, and alpha-compositing it onto every frame - no per-frame vector drawing,
and registration is exact by construction.

Drawn with PIL rather than matplotlib: it needs no DPI or bounding-box
arithmetic to line up, and it is faster than driving a vector renderer several
hundred times. Cartopy is still used, but only to *fetch* the Natural Earth
shapefile.

In [ ]:
overlays = {}
for p in cfg.products:
    path = cfg.overlay_path(p.grid)
    overlays[p.grid.name] = R.coastline_overlay(p.grid, path)
    h, w = p.grid.shape
    print(f"{p.grid.name:16s} {w:5d} x {h:4d} px   {p.grid.resolution}deg/px   -> {overlays[p.grid.name]}")

## 3. One timeslot, all composites

Render a single scene first and look at it. Getting this cell right is much
cheaper than discovering a problem 300 frames into the full run.

In [ ]:
from datetime import datetime
from IPython.display import Image as IPyImage, display

preview_time = datetime(2026, 9, 5, 3, 0)      # 10:00 WIB, plume well developed

for p in cfg.products:
    out = R.render_timeslot(
        preview_time, cfg.cache_dir, cfg.frames_dir, [p.composite], p.grid,
        overlay_path=overlays[p.grid.name],
        resample_cache=cfg.resample_cache,
        marker=cfg.marker, tz=cfg.tz, tz_label=cfg.tz_label,
        overwrite=True,
    )
    path = out.get(p.composite)
    print(f"\n{p.composite}  ->  {path}")
    if path:
        display(IPyImage(filename=str(path), width=900))

### Check it against a night scene too

This is the point of the IR products. At 18:00 UTC the box is entirely dark,
`true_color` is correctly skipped, and the ash RGB still shows the plume.

In [ ]:
night = datetime(2026, 9, 5, 18, 0)
ash = next(p for p in cfg.products if p.composite == "ash")

print("daylight anywhere in the box?", R.is_daylight(night, ash.grid))
out = R.render_timeslot(
    night, cfg.cache_dir, cfg.frames_dir, ["ash", "true_color"], ash.grid,
    overlay_path=overlays[ash.grid.name],
    resample_cache=cfg.resample_cache, marker=cfg.marker, tz=cfg.tz, tz_label=cfg.tz_label, overwrite=True,
)
print({k: ("rendered" if v else "skipped - dark") for k, v in out.items()})
display(IPyImage(filename=str(out["ash"]), width=900))

## 4. The full run

Three things keep this from taking all day, all of them measured on this
machine at 2026-09-05 03:00 UTC:

| | |
|---|---|
| `pad_data=False` | without it satpy pads the absent segments with NaN out to the whole 5500x5500 disk (22000x22000 for B03), so a 3-segment subset would carry the compute cost of the entire hemisphere |
| crop before resample | segments 5-7 span the full disk width; cropping to the target box first means the lazy composite arithmetic only runs on pixels that survive. True colour 157s -> 91s, output pixel-for-pixel identical |
| composite grouping | anything needing the reflective bands drags in B03, whose three segments decompress to ~290 MB. Keeping IR products in their own Scene: 15s for four IR composites vs 65s for true colour alone |

Plus the day/night gate, which is worth more than any of them: rendering true
colour over a dark box costs a minute to produce a black rectangle, and over
three days that is roughly half the timeslots.

Run the resample cache warm first (one timeslot, one worker) before turning
`render.workers` up - started cold, every worker computes the same KD-tree
simultaneously and they race to write the same cache file.

In [ ]:
from himawari.fetch import complete_times
from himawari import catalog

tasks = catalog.frame_tasks(cfg.timeslots(), cfg.bands, cfg.segments(), cfg.satellite)
times, incomplete = complete_times(tasks, cfg.cache_dir)
print(f"{len(times)} complete timeslots to render, {len(incomplete)} skipped as incomplete")

# Warm the resample cache: one slot, one worker, one KD-tree per grid.
for p in cfg.products:
    R.render_many(times[:1], cfg.cache_dir, cfg.frames_dir, [p.composite], p.grid,
                  overlay_path=overlays[p.grid.name], marker=cfg.marker, tz=cfg.tz, tz_label=cfg.tz_label,
                  resample_cache=cfg.resample_cache, workers=1, progress=False)
print("resample cache warm")

In [ ]:
import time

for p in cfg.products:
    t0 = time.time()
    print(f"\n=== {p.composite} on {p.grid.name} ===")
    R.render_many(
        times, cfg.cache_dir, cfg.frames_dir, [p.composite], p.grid,
        overlay_path=overlays[p.grid.name],
        resample_cache=cfg.resample_cache,
        marker=cfg.marker, tz=cfg.tz, tz_label=cfg.tz_label,
        workers=cfg.render_workers,
        skip_dark=cfg.skip_dark,
    )
    n = len(R.rendered_frames(cfg.frames_dir, p.composite, p.grid))
    print(f"{n} frames in {(time.time() - t0)/60:.1f} min")

## 5. What came out

Daylight-only composites will legitimately have far fewer frames than the
24-hour ones. Anything else missing is worth looking into before animating.

In [ ]:
from himawari.animate import frame_time

for p in cfg.products:
    frames = R.rendered_frames(cfg.frames_dir, p.composite, p.grid)
    have = {frame_time(f) for f in frames}
    gaps = [t for t in times if t not in have]
    daylight_only = R.needs_sunlight(p.composite)
    print(f"{p.composite:20s} {len(frames):4d} frames   "
          f"{len(gaps):4d} timeslots without one"
          + ("  (expected - daylight only)" if daylight_only else ""))
    if gaps and not daylight_only:
        for t in gaps[:5]:
            print(f"      {t:%Y-%m-%d %H:%M} UTC")

---

Next: **`03_animate.ipynb`** encodes these frames to MP4 and GIF.